# Structured Output과 Output Parser

Gemini 응답을 문자열, JSON(dict), Pydantic 객체로 받는 방법을 비교합니다.

In [1]:
from typing import Literal

from dotenv import load_dotenv
from langchain_core.exceptions import OutputParserException
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.output_parsers import (
    CommaSeparatedListOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
    StrOutputParser,
)
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

## Output Parser란?

Output Parser를 체인의 마지막에 연결하면 LLM 응답을 필요한 Python 타입으로 바꿀 수 있습니다.

| 방법 | 결과 타입 | 사용 시점 |
|---|---|---|
| `StrOutputParser` | `str` | 일반 텍스트만 필요할 때 |
| `CommaSeparatedListOutputParser` | `list[str]` | 단순 문자열 목록이 필요할 때 |
| `JsonOutputParser` | `dict` 또는 `list` | 유연한 JSON 데이터가 필요할 때 |
| `PydanticOutputParser` | Pydantic 객체 | JSON 파싱과 스키마 검증이 필요할 때 |
| `with_structured_output()` | Pydantic 객체 등 | Gemini의 네이티브 구조화 출력을 사용할 때 |

Parser 방식은 `get_format_instructions()`로 출력 규칙을 만든 뒤 프롬프트에 넣습니다. 이는 모델에게 형식을 **강제**하는 기능이 아니라 자연어 지시로 형식 준수를 **유도**하는 방식입니다. 모델은 설명 문장을 덧붙이거나 필드를 누락하는 등 지시를 어길 수 있습니다.

- 프롬프트와 `get_format_instructions()`: 원하는 출력 형식을 안내합니다.
- Output Parser: 실제 응답을 변환하고 형식 또는 schema 위반을 검출합니다.
- `with_structured_output()`: 스키마를 모델 API에 직접 전달합니다.

따라서 parser가 형식 준수를 보장하는 것은 아닙니다. 모델이 잘못된 결과를 반환하면 파싱 또는 validation 오류가 발생하며, 애플리케이션에서 이 실패를 처리해야 합니다. 지원 모델에서는 `with_structured_output()`이 일반적으로 더 간결하고 안정적이지만, 업무적으로 올바른 값까지 보장하지는 않습니다.

### 구조화 출력의 검증 단계

JSON으로 변환됐다고 해서 애플리케이션에서 사용할 수 있는 올바른 데이터라는 뜻은 아닙니다.

```text
JSON 문법 검증
→ schema 검증
→ domain 규칙 검증
→ policy 검증
```

| 검증 단계 | 확인 내용 | 실패 예 |
|---|---|---|
| JSON 문법 | 올바른 JSON인가 | 따옴표나 중괄호 누락 |
| Schema | 필드와 타입이 맞는가 | `intensity` 필드 누락 |
| Domain | 업무 범위에 맞는 값인가 | 감정 강도가 1~5 범위를 벗어남 |
| Policy | 서비스 정책상 허용되는가 | 근거 없이 위험한 판단을 확정함 |

Output Parser와 Pydantic은 주로 JSON 문법과 schema를 검증합니다. 업무 규칙이나 서비스 정책은 별도의 validator와 애플리케이션 로직으로 확인해야 합니다.

## PromptTemplate.partial()

Parser 예제의 `.partial()`은 프롬프트 변수 일부를 미리 채워 새로운 PromptTemplate을 만드는 메서드입니다. Python의 `functools.partial()`처럼 매번 바뀌지 않는 값을 사전에 바인딩한다고 생각하면 됩니다.

```python
prompt = PromptTemplate.from_template(
    "입력: {text}\n{format_instructions}"
)

prompt = prompt.partial(
    format_instructions=parser.get_format_instructions()
)

# format_instructions는 이미 채워졌으므로 실행할 때 text만 전달합니다.
prompt.invoke({"text": "분석할 문장"})
```

여기서 `format_instructions`는 parser가 정한 고정 규칙이고 `text`는 호출마다 달라지는 사용자 입력입니다. `.partial()`을 사용하지 않는다면 `invoke()`를 호출할 때 두 값을 모두 전달해야 합니다.

`.partial()`은 문자열을 즉시 완성하거나 LLM을 호출하지 않습니다. 아직 채워지지 않은 변수만 입력으로 받는 새로운 프롬프트 템플릿을 반환합니다.

## StrOutputParser

Gemini의 `AIMessage`에서 텍스트만 꺼내는 가장 기본적인 parser입니다.

In [2]:
text_prompt = ChatPromptTemplate.from_template(
    "{topic}을 처음 배우는 사람에게 두 문장으로 설명해 주세요."
)
text_chain = text_prompt | llm | StrOutputParser()
text_result = text_chain.invoke({"topic": "LangChain"})

print(type(text_result).__name__)
print(text_result)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


TextAccessor
LangChain은 인공지능(LLM)을 활용하여 원하는 기능을 가진 서비스를 쉽게 만들 수 있도록 돕는 도구입니다. 이것은 LLM이 외부 데이터나 다른 프로그램과 연결되어 더 똑똑하고 유용하게 작동하도록 만들어줍니다.


## CommaSeparatedListOutputParser

쉼표로 구분된 짧은 목록을 `list[str]`로 바꿉니다. 태그나 키워드처럼 단순한 1차원 목록에 적합합니다. 항목 안에 쉼표가 들어가거나 구조가 복잡해지면 `JsonOutputParser`를 사용하세요.

In [3]:
list_parser = CommaSeparatedListOutputParser()
list_prompt = PromptTemplate.from_template(
    "{topic}과 관련된 핵심 키워드 5개를 작성해 주세요.\n{format_instructions}"
).partial(format_instructions=list_parser.get_format_instructions())

list_chain = list_prompt | llm | list_parser
list_result = list_chain.invoke({"topic": "생성형 AI"})

print(type(list_result).__name__)
print(list_result)

list
['생성', '모델', '데이터', '학습', '콘텐츠']


## JsonOutputParser

고정된 모델까지는 필요 없지만 JSON을 Python `dict`로 받고 싶을 때 사용합니다.

In [4]:
json_parser = JsonOutputParser()
json_prompt = PromptTemplate.from_template(
    """다음 주제의 핵심 개념 3개를 JSON으로 정리해 주세요.
주제: {topic}
{format_instructions}"""
).partial(format_instructions=json_parser.get_format_instructions())

json_chain = json_prompt | llm | json_parser
json_result = json_chain.invoke({"topic": "RAG"})

print(type(json_result).__name__)
print(json_result)

dict
{'topic': 'RAG', 'core_concepts': [{'name': '검색 (Retrieval)', 'description': '사용자의 질문과 관련된 외부 지식 기반(예: 문서, 데이터베이스)에서 가장 관련성 높은 정보를 찾아내는 과정입니다. 주로 벡터 데이터베이스와 임베딩을 활용하여 의미론적 유사성을 기반으로 문서를 검색합니다.'}, {'name': '정보 증강 및 생성 (Information Augmentation & Generation)', 'description': '검색된 정보를 대규모 언어 모델(LLM)의 프롬프트에 추가하여, LLM이 이 추가된 컨텍스트를 기반으로 더 정확하고 사실적이며 최신 정보를 포함하는 답변을 생성하도록 하는 과정입니다. LLM의 환각(hallucination)을 줄이는 데 기여합니다.'}, {'name': '외부 지식 기반 (External Knowledge Base)', 'description': 'LLM이 학습하지 않은 최신 정보, 특정 도메인 지식, 사내 문서 등 다양한 형태의 외부 데이터를 저장하고 관리하는 저장소입니다. RAG 시스템이 정보를 검색할 수 있는 원천이 됩니다.'}]}


## PydanticOutputParser

LLM이 만든 JSON 텍스트를 파싱하고 Pydantic 스키마로 검증합니다. 필수 필드, 타입, 값의 범위 등을 적용할 수 있습니다.

In [5]:
class SentimentResult(BaseModel):
    sentiment: Literal["긍정", "부정", "중립"] = Field(description="문장에서 나타나는 감정")
    intensity: int = Field(ge=1, le=5, description="감정 강도")
    reason: str = Field(min_length=1, description="판단 근거")


pydantic_parser = PydanticOutputParser(pydantic_object=SentimentResult)
pydantic_prompt = PromptTemplate.from_template(
    """다음 문장의 감정을 분석해 주세요.
문장: {text}
{format_instructions}"""
).partial(format_instructions=pydantic_parser.get_format_instructions())

pydantic_chain = pydantic_prompt | llm | pydantic_parser
pydantic_result = pydantic_chain.invoke({"text": "새 기능이 기대보다 훨씬 좋아서 정말 만족합니다."})

print(type(pydantic_result).__name__)
print(pydantic_result)

SentimentResult
sentiment='긍정' intensity=5 reason="문장에서 '기대보다 훨씬 좋아서'라는 긍정적인 평가와 '정말 만족합니다'라는 강한 만족감을 직접적으로 표현하고 있어 매우 긍정적인 감정을 나타냅니다."


### 설명이 아니라 타입으로 허용값 제한하기

필드 설명은 모델에게 원하는 값을 안내하지만 실제 허용값을 제한하지는 않습니다. `sentiment: str`을 사용하면 `"만족"`, `"혼합"`, `"알 수 없음"` 같은 값도 문자열이므로 검증을 통과합니다.

`Literal["긍정", "부정", "중립"]`처럼 허용값을 타입에 명시하면 Pydantic이 그 밖의 값을 검증 단계에서 차단합니다. 숫자는 `ge`, `le`, 문자열은 `min_length`처럼 데이터 특성에 맞는 제약을 함께 지정할 수 있습니다.

## Gemini Structured Output

`with_structured_output()`은 Pydantic 스키마를 Gemini에 직접 전달합니다. 별도 format instructions나 parser 없이 검증된 객체를 바로 받습니다.

In [6]:
structured_llm = llm.with_structured_output(SentimentResult)
structured_result = structured_llm.invoke(
    "배송은 빨랐지만 제품 포장이 찢어져 있어서 아쉬웠습니다."
)

print(type(structured_result).__name__)
print(structured_result)

SentimentResult
sentiment='부정' intensity=4 reason="배송이 빨랐다는 긍정적인 내용이 있지만, 제품 포장이 찢어져 있었다는 부정적인 내용과 '아쉬웠습니다'라는 표현으로 인해 전반적으로 부정적인 감정이 나타납니다."


### PydanticOutputParser를 사용하는 경우

모델이 네이티브 structured output을 안정적으로 지원한다면 `with_structured_output()`을 우선 고려할 수 있습니다. 하지만 다음과 같은 상황에서는 `PydanticOutputParser`가 더 적합합니다.

- 사용하는 provider나 모델이 네이티브 structured output을 지원하지 않는 경우
- 여러 provider에서 같은 프롬프트와 파싱 방식을 유지해야 하는 경우
- 이미 저장된 JSON 문자열이나 외부에서 받은 텍스트를 Pydantic 객체로 검증해야 하는 경우
- 모델의 원본 텍스트를 전처리한 뒤 직접 파싱하거나 파싱 실패 과정을 세밀하게 제어해야 하는 경우
- `FakeListChatModel`처럼 출력이 문자열인 테스트 모델로 성공과 실패를 재현해야 하는 경우

| 구분 | `with_structured_output()` | `PydanticOutputParser` |
|---|---|---|
| 구조 전달 방식 | 스키마를 모델 API에 전달 | format instructions를 프롬프트에 전달 |
| 모델 요구사항 | 해당 모델의 구조화 출력 지원 필요 | 일반 텍스트 출력 모델에서도 사용 가능 |
| 검증 시점 | 모델 호출 과정과 결합 | 문자열 응답을 받은 뒤 파싱·검증 |
| 제어 범위 | 간결하지만 provider 동작에 영향을 받음 | 전처리, 예외 처리, 재시도 흐름을 직접 구성 가능 |

`PydanticOutputParser`는 프롬프트로 형식을 안내하므로 모델이 JSON 형식을 지키지 않을 수 있습니다. 따라서 네이티브 structured output의 단순한 하위 호환 기능이라기보다, 문자열 응답을 직접 다뤄야 할 때 선택하는 별도의 실행 방식으로 이해합니다.

---

## Structured Output이 실패할 때

실패 원인은 크게 세 가지이며, 실패 위치에 따라 처리 방법이 달라집니다.

- **Provider/API 실패**: 모델이 네이티브 structured output을 지원하지 않거나 API 요청 자체가 실패한 경우입니다. 원본 응답이 없으므로 예외가 발생합니다.
- **파싱 실패**: 응답은 받았지만 JSON 문법이나 필요한 구조가 잘못된 경우입니다.
- **검증 실패**: JSON 구조는 맞지만 Pydantic 타입, 필수 필드 또는 범위 조건을 위반한 경우입니다.

먼저 원본 응답과 파싱 오류를 함께 보면서 원인을 구분해야 합니다. Gemini의 `with_structured_output(..., include_raw=True)`는 `raw`, `parsed`, `parsing_error`를 함께 반환하므로 디버깅에 유용합니다.

주의할 점은 `include_raw=True`일 때 파싱·검증 실패가 예외가 아니라 `parsing_error` 값으로 반환될 수 있다는 것입니다. `.with_retry()`와 `.with_fallbacks()`는 기본적으로 **예외가 발생해야** 동작하므로, 이 진단용 결과를 그대로 retry/fallback 체인에 연결하면 자동 복구가 실행되지 않을 수 있습니다. 진단 모드와 운영 복구 체인은 분리하는 편이 명확합니다.

In [7]:
diagnostic_llm = llm.with_structured_output(SentimentResult, include_raw=True)
diagnostic_result = diagnostic_llm.invoke(
    "생각보다 나쁘지는 않았지만 다시 구매할지는 모르겠습니다."
)

print("parsed:", diagnostic_result["parsed"])
print("parsing_error:", diagnostic_result["parsing_error"])
print("raw:", diagnostic_result["raw"])

parsed: sentiment='중립' intensity=2 reason='생각보다 나쁘지 않았지만 재구매 의사가 확실하지 않아 긍정적이지도 부정적이지도 않은 중립적인 감정입니다.'
parsing_error: None
raw: content='{"sentiment": "중립", "intensity": 2, "reason": "생각보다 나쁘지 않았지만 재구매 의사가 확실하지 않아 긍정적이지도 부정적이지도 않은 중립적인 감정입니다."}' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a05f99-95a1-7303-aeef-fe58c4d35630-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 16, 'output_tokens': 603, 'total_tokens': 619, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 552}}


### 재시도

이 예제는 네이티브 structured output이 아니라 **프롬프트 + 일반 `PydanticOutputParser` 방식의 재시도**를 보여줍니다. 실제 LLM이 우연히 잘못된 응답을 만들기를 기다리면 실행할 때마다 결과가 달라지므로 `FakeListChatModel`에 응답을 미리 지정합니다. 첫 번째 호출은 Pydantic 검증에 실패하고 두 번째 호출은 성공합니다.

실행 흐름은 다음과 같습니다.

```text
첫 번째 호출: intensity=10 → Pydantic 범위 검증 실패 → OutputParserException
두 번째 호출: intensity=4  → Pydantic 검증 성공 → SentimentResult 반환
```

`.with_retry()`는 같은 문자열에 parser만 다시 적용하지 않습니다. `RunnableLambda | prompt | fake_llm | parser` 전체를 다시 실행하여 모델로부터 새로운 응답을 받습니다. `retry_if_exception_type`으로 재시도할 오류를 제한하고, `stop_after_attempt`로 무한 반복을 막습니다. 실제 운영에서는 `fake_llm` 자리에 Gemini를 사용합니다.

In [8]:
attempts = {"count": 0}

def count_attempts(value):
    attempts["count"] += 1
    print(f"시도 {attempts['count']}회")
    return value


fake_retry_llm = FakeListChatModel(
    responses=[
        # 첫 번째 응답: intensity가 허용 범위(1~5)를 벗어납니다.
        '{"sentiment": "긍정", "intensity": 10, "reason": "만족한다고 표현했습니다."}',
        # 두 번째 응답: 검증을 통과합니다.
        '{"sentiment": "긍정", "intensity": 4, "reason": "만족한다고 표현했습니다."}',
    ]
)

retry_demo_chain = (
    RunnableLambda(count_attempts)
    | pydantic_prompt
    | fake_retry_llm
    | pydantic_parser
).with_retry(
    retry_if_exception_type=(OutputParserException,),
    stop_after_attempt=3,
)

retry_result = retry_demo_chain.invoke({"text": "이 제품은 정말 좋습니다."})
print("총 시도 횟수:", attempts["count"])
print("최종 결과:", retry_result)

시도 1회
시도 2회
총 시도 횟수: 2
최종 결과: sentiment='긍정' intensity=4 reason='만족한다고 표현했습니다.'


### Fallback

Fallback 예제는 **네이티브 structured output 실패 → 일반 parser 방식 전환**을 단순화해서 보여줍니다. 실제 장애를 기다리지 않고 primary Runnable에서 의도적으로 `RuntimeError`를 발생시킵니다. Primary가 정상 값을 반환하면 fallback은 실행되지 않으며, 예외가 밖으로 전달될 때만 `.with_fallbacks()`가 같은 입력을 다음 체인에 전달합니다.

```text
primary_chain 실행 → RuntimeError
같은 {text} 입력을 fallback_chain에 전달
Fake 모델의 JSON 문자열 → PydanticOutputParser 검증 → 성공 객체 반환
```

여기서 primary는 실패 재현을 위한 가짜 Runnable이고 fallback은 일반 parser 체인입니다. 실제 운영에서는 primary에 `llm.with_structured_output(Schema)`를 연결하고, fallback에는 parser 기반 체인이나 다른 모델을 연결할 수 있습니다. 인증 오류나 정책 위반처럼 전환해도 해결되지 않는 오류까지 무조건 fallback하지 않도록 대상 예외를 구분해야 합니다.

In [9]:
def raise_structured_output_error(_):
    print("primary 실행: 의도적인 실패")
    raise RuntimeError("네이티브 structured output을 사용할 수 없습니다.")


fake_fallback_llm = FakeListChatModel(
    responses=[
        '{"sentiment": "중립", "intensity": 2, "reason": "장점과 단점이 함께 언급되었습니다."}'
    ]
)

primary_chain = RunnableLambda(raise_structured_output_error)
fallback_chain = pydantic_prompt | fake_fallback_llm | pydantic_parser
resilient_chain = primary_chain.with_fallbacks([fallback_chain])

fallback_result = resilient_chain.invoke(
    {"text": "속도는 빨라졌지만 오류가 더 자주 발생합니다."}
)
print("fallback 결과:", fallback_result)

primary 실행: 의도적인 실패
fallback 결과: sentiment='중립' intensity=2 reason='장점과 단점이 함께 언급되었습니다.'


### 모든 실패를 재시도하지는 않는다

구조화 출력이 실패했다고 해서 항상 같은 요청을 반복하면 안 됩니다. 실패 원인에 따라 처리 방법을 구분합니다.

| 실패 상황 | 권장 처리 |
|---|---|
| JSON 형식이 일시적으로 잘못됨 | 횟수를 제한하여 재시도 |
| 필수 입력이 부족함 | 사용자에게 추가 정보 요청 |
| 허용 범위를 벗어난 값 | 오류를 알리고 중단하거나 다시 입력받기 |
| 정책상 허용되지 않는 결과 | 재시도하지 않고 중단 |
| 일시적인 provider 장애 | 조건에 따라 fallback 모델 사용 |

현재 예제의 재시도는 재현 가능한 형식·검증 실패를 대상으로 합니다. 인증 오류, 잘못된 사용자 입력, 정책 위반처럼 반복해도 해결되지 않는 오류는 재시도 대상에서 제외해야 합니다. 재시도에는 반드시 최대 횟수를 설정하여 무한 반복을 막습니다.

## 선택 기준

- 단순 텍스트: `StrOutputParser`
- 유연한 JSON: `JsonOutputParser`
- 프롬프트 기반 JSON 파싱과 엄격한 검증: `PydanticOutputParser`
- 모델이 지원하고 스키마가 명확한 경우: `with_structured_output()`

운영 환경에서는 파싱·검증 실패에 대비해 예외 처리와 재시도도 구성합니다. 예: `chain.with_retry(stop_after_attempt=3)`.